In [7]:
#r "./bin/Debug/net10.0/task14.dll"
using task14;
using System;
using System.Diagnostics;
using System.Collections.Generic;
using System.Linq;
using System.IO;

var SIN = (double x) => Math.Sin(x);
double a = -100.0;
double b = 100.0;
double precision = 1e-4;
double value = 0.0;

In [8]:
double[] steps = {1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6};
var stepres = new List<(double step, double error, double time)>();
foreach (var step in steps)
{
    double time = 0;
    double averageError = 0;
    int iterations = 10; 
    for (int i = 0; i < iterations; i++)
    {
        var stopwatch = Stopwatch.StartNew();
        double res = DefiniteIntegral.Solve(a, b, SIN, step, Environment.ProcessorCount);
        stopwatch.Stop();
        time += stopwatch.Elapsed.TotalMilliseconds;
        averageError += Math.Abs(res - value);
    }
    double averagetime = time / iterations;
    double error = averageError / iterations;
    stepres.Add((step, error, averagetime));
    Console.WriteLine($"Шаг: {step:E1}, Ошибка: {error:E4}, Время: {averagetime:F2}");
}
var optimstep = stepres.Where(a => a.error <= precision).OrderByDescending(a => a.step).FirstOrDefault();
Console.WriteLine($"Оптимальный шаг: {optimstep.step:E1}");

Шаг: 1,0E-001, Ошибка: 1,2143E-015, Время: 0,03
Шаг: 1,0E-002, Ошибка: 3,2474E-015, Время: 0,12
Шаг: 1,0E-003, Ошибка: 1,6862E-015, Время: 0,32
Шаг: 1,0E-004, Ошибка: 1,7583E-014, Время: 3,38
Шаг: 1,0E-005, Ошибка: 2,6784E-014, Время: 31,51
Шаг: 1,0E-006, Ошибка: 3,7165E-014, Время: 307,56
Оптимальный шаг: 1,0E-001


In [9]:
double selectstep = optimstep.step;
int[] threadcount = Enumerable.Range(1, 16).ToArray();
var threadres = new List<(int threads, double time)>();
foreach (var tc in threadcount)
{
    double time = 0;
    int iterations = 10;

    for (int i = 0; i < iterations; i++)
    {
        var stopwatch = Stopwatch.StartNew();
        DefiniteIntegral.Solve(a, b, SIN, selectstep, tc);
        stopwatch.Stop();
        time += stopwatch.Elapsed.TotalMilliseconds;
    }
    double averagetime = time / iterations;
    threadres.Add((tc, averagetime));
    Console.WriteLine($"Потоков: {tc,2} - Время: {averagetime:F2} мс");
}
var optimthreads = threadres.OrderBy(r => r.time).FirstOrDefault();
Console.WriteLine($"Оптимальное количество потоков: {optimthreads.threads}");

Потоков:  1 - Время: 0,06 мс
Потоков:  2 - Время: 0,04 мс
Потоков:  3 - Время: 0,05 мс
Потоков:  4 - Время: 0,04 мс
Потоков:  5 - Время: 0,03 мс
Потоков:  6 - Время: 0,02 мс
Потоков:  7 - Время: 0,02 мс
Потоков:  8 - Время: 0,02 мс
Потоков:  9 - Время: 0,02 мс
Потоков: 10 - Время: 0,02 мс
Потоков: 11 - Время: 0,01 мс
Потоков: 12 - Время: 0,01 мс
Потоков: 13 - Время: 0,01 мс
Потоков: 14 - Время: 0,02 мс
Потоков: 15 - Время: 0,01 мс
Потоков: 16 - Время: 0,01 мс
Оптимальное количество потоков: 12


In [10]:
#r "nuget: ScottPlot, 5.1.59"

var graphic = new ScottPlot.Plot();
double[] xValues = threadres.Select(r => (double)r.threads).ToArray();
double[] yValues = threadres.Select(r => r.time).ToArray();
var scatter = graphic.Add.Scatter(xValues, yValues);
scatter.Color = ScottPlot.Colors.Pink;
scatter.LineWidth = 5;
scatter.MarkerSize = 10;
scatter.MarkerShape = ScottPlot.MarkerShape.FilledCircle;

graphic.Title("Время вычисления от количества потоков");
graphic.XLabel("Количество потоков");
graphic.YLabel("Время");

graphic.Grid.MajorLineColor = ScottPlot.Colors.Gray.WithOpacity(0.3);
graphic.SavePng("result_on_graphic.png", 1000, 500);

Installed Packages ScottPlot, 5.1.59

In [11]:
int iterations = 40;
double onethreadtime = 0;
for (int i = 0; i < iterations; i++)
{
    var stopwatch = Stopwatch.StartNew();
    OneThreadIntegral.Solve(a, b, SIN, selectstep);
    stopwatch.Stop();
    onethreadtime += stopwatch.Elapsed.TotalMilliseconds;
}
onethreadtime /= iterations;
double manythreadstime = 0;
for (int i = 0; i < iterations; i++)
{
    var stopwatch = Stopwatch.StartNew();
    DefiniteIntegral.Solve(a, b, SIN, selectstep, optimthreads.threads);
    stopwatch.Stop();
    manythreadstime += stopwatch.Elapsed.TotalMilliseconds;
}
manythreadstime /= iterations;
double speed = onethreadtime / manythreadstime;
double percent = ((onethreadtime - manythreadstime) / onethreadtime) * 100;
Console.WriteLine($"Однопоточное вычисление: {onethreadtime:F2} мс");
Console.WriteLine($"Многопоточное вычисление: {manythreadstime:F2} мс");
Console.WriteLine($"Ускорение: {speed:F2}");
Console.WriteLine($"Разница: {percent:F2}%");
if (percent >= 15)
    Console.WriteLine("Многопоточное вычисление быстрее на 15% и более");
else
    Console.WriteLine("Разница менее 15%, требуется оптимизация");

Однопоточное вычисление: 0,02 мс
Многопоточное вычисление: 0,02 мс
Ускорение: 1,46
Разница: 31,63%
Многопоточное вычисление быстрее на 15% и более


In [12]:
// Пункт 9: Записываем результаты в файл
string information = $@"
1. Оптимальный размер шага: {optimstep.step:E1}
- Шаг показывает, насколько мелкие отрезки используются при вычислении интеграла.
2. Оптимальное количество потоков: {optimthreads.threads}
- Число параллельных потоков, которое даёт минимальное время ({optimthreads.time:F2} мс) выполнения.
3. Сравнение однопоточного с многопоточным вычислением:
- Время однопоточного вычисления: {onethreadtime:F2} мс - среднее время последовательного вычисления интеграла.
- Время многопоточного вычисления: {manythreadstime:F2} мс - среднее время параллельного вычисления с {optimthreads.threads} потоками.
- Ускорение: {speed:F2} - во столько раз многопоточное вычисление быстрее однопоточного.
- Разница в процентах: {percent:F2}% - ускорение в процентах.
";
File.WriteAllText("results_in_text.txt", information);
Console.WriteLine(information);


1. Оптимальный размер шага: 1,0E-001
- Шаг показывает, насколько мелкие отрезки используются при вычислении интеграла.
2. Оптимальное количество потоков: 12
- Число параллельных потоков, которое даёт минимальное время (0,01 мс) выполнения.
3. Сравнение однопоточного с многопоточным вычислением:
- Время однопоточного вычисления: 0,02 мс - среднее время последовательного вычисления интеграла.
- Время многопоточного вычисления: 0,02 мс - среднее время параллельного вычисления с 12 потоками.
- Ускорение: 1,46 - во столько раз многопоточное вычисление быстрее однопоточного.
- Разница в процентах: 31,63% - ускорение в процентах.

